In [ ]:

import sys
from src.api import DadosAbertosPBAPI
from src.extractor import BaseExtractor
from typing import Any, Dict, List
from pathlib import Path
import os
import subprocess
from datetime import datetime
import json
import traceback
import time

api = DadosAbertosPBAPI()

In [2]:
class AtasExtractor(BaseExtractor):
    @property
    def resource_name(self) -> str:
        return "atas"

    @property
    def period_mode(self) -> str:
        return "ano_inicio_vigencia"

    def fetch_vigencia_records(self, year: int) -> List[Dict[str, Any]]:
        return self.api.obter_todas_atas_registro_preco(
            ano_inicio_vigencia=year
        )


class NotasEmpenhoExtractor(BaseExtractor):
    @property
    def resource_name(self) -> str:
        return "notas_empenho"

    def fetch_month_records(self, year: int, month: int) -> List[Dict[str, Any]]:
        return self.api.obter_todas_notas_empenho(
            ano=year,
            mes=month
        )


class LiquidacoesExtractor(BaseExtractor):
    @property
    def resource_name(self) -> str:
        return "liquidacoes"

    def fetch_month_records(self, year: int, month: int) -> List[Dict[str, Any]]:
        return self.api.obter_todas_liquidacoes(
            ano=year,
            mes=month
        )


class ContratosExtractor(BaseExtractor):
    @property
    def resource_name(self) -> str:
        return "contratos"

    def fetch_year_records(self, year: int) -> List[Dict[str, Any]]:
        return self.api.obter_todos_contratos(
            ano_inicio_vigencia=year
        )


class ContratacoesExtractor(BaseExtractor):
    @property
    def resource_name(self) -> str:
        return "contratacoes"

    def fetch_month_records(self, year: int, month: int) -> List[Dict[str, Any]]:
        return self.api.obter_todas_contratacoes(
            ano=year,
            mes=month
        )


class ServidoresExtractor(BaseExtractor):
    @property
    def resource_name(self) -> str:
        return "servidores"

    def fetch_month_records(self, year: int, month: int) -> List[Dict[str, Any]]:
        return self.api.obter_todos_servidores(
            ano=year,
            mes=month
        )

In [ ]:
ANO_INICIAL = 2025
ANO_FINAL = datetime.now().year

ANOS = range(ANO_INICIAL, ANO_FINAL + 1)

FORMATOS = ["csv"]
OUTPUT_DIR = Path("./data/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DIRETORIO_DADOS = str(OUTPUT_DIR)

extratores = {
    "atas": AtasExtractor,
    "notas_empenho": NotasEmpenhoExtractor,
    "liquidacoes": LiquidacoesExtractor,
    "contratos": ContratosExtractor,
    "contratacoes": ContratacoesExtractor,
    "servidores": ServidoresExtractor,
}

relatorio_geral = []


for nome, classe_extrator in extratores.items():

    print("\n" + "=" * 70)
    print(f"RECURSO: {nome}")
    print("=" * 70)

    for ano in ANOS:

        print(f"\nExtraindo {nome} de {ano}...")

        try:
            extrator = classe_extrator(
                api=api,
                output_dir=DIRETORIO_DADOS,
            )

            resultado = extrator.extract_and_save_year(
                ano,
                formats=FORMATOS,
                include_report=True,
            )

            registro = {
                "recurso": nome,
                "ano": ano,
                "status": "sucesso",
                "total_registros": resultado.get("total_registros", 0),
                "arquivos": resultado.get("arquivos", []),
            }

            print(f"Registros: {registro['total_registros']}")
            print(f"Arquivos: {registro['arquivos']}")

        except Exception as erro:

            registro = {
                "recurso": nome,
                "ano": ano,
                "status": "erro",
                "erro": str(erro),
            }

            print(f"ERRO em {nome}, ano {ano}: {erro}")
            traceback.print_exc()

        relatorio_geral.append(registro)

        time.sleep(1)


caminho_relatorio = OUTPUT_DIR / "relatorio_extracao.json"

with open(caminho_relatorio, "w", encoding="utf-8") as arquivo:
    json.dump(
        relatorio_geral,
        arquivo,
        ensure_ascii=False,
        indent=2,
        default=str,
    )


sucessos = sum(
    item["status"] == "sucesso"
    for item in relatorio_geral
)

erros = sum(
    item["status"] == "erro"
    for item in relatorio_geral
)

total_registros = sum(
    item.get("total_registros", 0)
    for item in relatorio_geral
)

print("\n" + "=" * 70)
print("RESUMO DA EXTRAÇÃO")
print("=" * 70)

print(f"Execuções bem-sucedidas: {sucessos}")
print(f"Execuções com erro: {erros}")
print(f"Total de registros reportados: {total_registros}")
print(f"Relatório: {caminho_relatorio}")
print(f"Dados armazenados em: {OUTPUT_DIR}")

2026-10-28 20:34:32,913 - src.extractor - INFO - 
2026-10-28 20:34:32,914 - src.extractor - INFO - Processando ano 2025
2026-10-28 20:34:32,914 - src.extractor - INFO - ============================================================



RECURSO: atas

Extraindo atas de 2025...


2026-10-28 20:34:35,691 - src.extractor - INFO - Total para 2025: 2500 registros
2026-10-28 20:34:35,776 - src.extractor - INFO - CSV salvo: data/output/atas_2025.csv
2026-10-28 20:34:35,832 - src.extractor - INFO - Relatório salvo: data/output/atas_2025_relatorio.txt


Registros: 2500
Arquivos: {'csv': PosixPath('data/output/atas_2025.csv')}


2026-10-28 20:34:36,834 - src.extractor - INFO - 
2026-10-28 20:34:36,835 - src.extractor - INFO - Processando ano 2026
2026-10-28 20:34:36,835 - src.extractor - INFO - ============================================================


...

2026-10-28 20:54:32,192 - src.api - INFO - Página 1/1357 - Total acumulado: 100 registros
2026-10-28 20:54:32,352 - src.api - INFO - Página 2/1357 - Total acumulado: 200 registros
2026-10-28 20:54:32,510 - src.api - INFO - Página 3/1357 - Total acumulado: 300 registros
2026-10-28 20:54:32,662 - src.api - INFO - Página 4/1357 - Total acumulado: 400 registros
2026-10-28 20:54:32,821 - src.api - INFO - Página 5/1357 - Total acumulado: 500 registros
2026-10-28 20:54:32,988 - src.api - INFO - Página 6/1357 - Total acumulado: 600 registros
2026-10-28 20:54:33,150 - src.api - INFO - Página 7/1357 - Total acumulado: 700 registros
2026-10-28 20:54:33,322 - src.api - INFO - Página 8/1357 - Total acumulado: 800 registros
2026-10-28 20:54:33,495 - src.api - INFO - Página 9/1357 - Total acumulado: 900 registros
2026-10-28 20:54:33,656 - src.api - INFO - Página 10/1357 - Total acumulado: 1000 registros
2026-10-28 20:54:33,831 - src.api - INFO - Página 11/1357 - Total acumulado: 1100 registros
2026-1

Registros: 1623534
Arquivos: {'csv': PosixPath('data/output/servidores_2025.csv')}


2026-10-28 22:02:09,892 - src.extractor - INFO - 
2026-10-28 22:02:09,894 - src.extractor - INFO - Processando ano 2026
2026-10-28 22:02:09,896 - src.extractor - INFO - ============================================================



Extraindo servidores de 2026...


2026-10-28 22:02:10,152 - src.api - INFO - Página 1/1335 - Total acumulado: 100 registros
2026-10-28 22:02:10,323 - src.api - INFO - Página 2/1335 - Total acumulado: 200 registros
2026-10-28 22:02:10,485 - src.api - INFO - Página 3/1335 - Total acumulado: 300 registros
2026-10-28 22:02:10,642 - src.api - INFO - Página 4/1335 - Total acumulado: 400 registros
2026-10-28 22:02:10,802 - src.api - INFO - Página 5/1335 - Total acumulado: 500 registros
2026-10-28 22:02:10,968 - src.api - INFO - Página 6/1335 - Total acumulado: 600 registros
2026-10-28 22:02:11,132 - src.api - INFO - Página 7/1335 - Total acumulado: 700 registros
2026-10-28 22:02:11,295 - src.api - INFO - Página 8/1335 - Total acumulado: 800 registros
2026-10-28 22:02:11,455 - src.api - INFO - Página 9/1335 - Total acumulado: 900 registros
2026-10-28 22:02:11,621 - src.api - INFO - Página 10/1335 - Total acumulado: 1000 registros
2026-10-28 22:02:11,786 - src.api - INFO - Página 11/1335 - Total acumulado: 1100 registros
2026-1